# Training Pipelines

This notebook implements the two training pipelines for the GPT model:

- **PretrainPipeline** — trains the model from scratch on raw text using next-token prediction (cross-entropy loss), with AdamW optimizer, linear warmup + cosine decay LR schedule, train/val splitting, checkpoint saving, and overfitting detection.
- **SFTPipeline** — fine-tunes a pretrained checkpoint on instruction-response pairs, computing loss only on output tokens (instruction tokens are masked).

### Key Design Decisions

1. **LR Schedule**: Linear warmup prevents early instability; cosine decay smoothly reduces LR to zero.
2. **Overfitting Detection**: If validation loss increases for `patience` consecutive evaluations, a warning is logged.
3. **SFT Loss Masking**: Only output tokens contribute to the loss — the model is not penalised for the instruction portion.
4. **Reproducibility**: Random seeds are set for Python, NumPy, and PyTorch (CPU + CUDA) at the start of each run.

In [ ]:
import logging
import math
import os
import random
from dataclasses import dataclass, asdict
from typing import Any

import numpy as np
import torch
import torch.nn as nn
from torch import Tensor

# For inline plots
import matplotlib.pyplot as plt
%matplotlib inline

logger = logging.getLogger(__name__)

---
## Part 1 — Pretraining Pipeline

The pretraining pipeline trains the GPT model on raw text using **next-token prediction**.
Given a sequence of tokens `[t₁, t₂, ..., tₙ]`, the model predicts `[t₂, t₃, ..., tₙ₊₁]`
and is trained with cross-entropy loss.

### Learning Rate Schedule

The LR schedule has two phases:

1. **Linear warmup** (steps 0 → `warmup_steps`): LR increases linearly from 0 to `peak_lr`.
   This prevents large, destabilising updates in the first few steps when the model
   weights are still random.

2. **Cosine decay** (steps `warmup_steps` → `max_steps`): LR decreases following a
   cosine curve from `peak_lr` to 0. This provides a smooth annealing that helps the
   model converge to a better minimum.

```
LR
 ^
 |        peak_lr
 |       /‾‾‾‾‾\
 |      /        \
 |     /          \  cosine decay
 |    / warmup     \
 |   /              \
 |  /                \
 | /                  \___
 +---------------------------→ step
 0    W                M
```

### `get_lr` — Learning Rate Schedule Function

Computes the learning rate for a given training step. During warmup, LR scales
linearly from 0 to `peak_lr`. After warmup, it follows cosine decay from
`peak_lr` to 0.

This function is called at every training step to update the optimizer's
learning rate (manual LR scheduling).

In [ ]:
def get_lr(step: int, warmup_steps: int, max_steps: int, peak_lr: float) -> float:
    """Compute learning rate for a given training step.

    Linear warmup from 0 to *peak_lr* over *warmup_steps*, then cosine
    decay from *peak_lr* to 0 over the remaining steps.

    Args:
        step: Current training step (0-indexed).
        warmup_steps: Number of warmup steps.
        max_steps: Total number of training steps.
        peak_lr: Peak learning rate reached at the end of warmup.

    Returns:
        Learning rate for the given step.
    """
    if max_steps <= 0:
        return 0.0
    if warmup_steps <= 0:
        # No warmup — pure cosine decay from peak_lr
        progress = min(step / max(max_steps, 1), 1.0)
        return peak_lr * 0.5 * (1.0 + math.cos(math.pi * progress))
    if step < warmup_steps:
        # Linear warmup
        return peak_lr * step / warmup_steps
    if step >= max_steps:
        return 0.0
    # Cosine decay
    decay_steps = max_steps - warmup_steps
    decay_progress = (step - warmup_steps) / decay_steps
    return peak_lr * 0.5 * (1.0 + math.cos(math.pi * decay_progress))

### Visualise the LR Schedule

Let's plot the learning rate over training steps to see the warmup + cosine
decay curve in action.

In [ ]:
# Visualise the LR schedule
warmup = 500
total = 10000
peak = 3e-4

steps = list(range(total))
lrs = [get_lr(s, warmup, total, peak) for s in steps]

plt.figure(figsize=(8, 3))
plt.plot(steps, lrs)
plt.xlabel('Step')
plt.ylabel('Learning Rate')
plt.title('Linear Warmup + Cosine Decay LR Schedule')
plt.axvline(x=warmup, color='r', linestyle='--', alpha=0.5, label=f'warmup={warmup}')
plt.legend()
plt.tight_layout()
plt.show()

### `detect_overfitting` — Overfitting Detection

Checks whether validation loss has been increasing for `patience` consecutive
evaluations. If so, it signals that the model is starting to overfit — it's
memorising the training data rather than learning generalisable patterns.

The function examines the last `patience + 1` validation losses and returns
`True` if each successive value is strictly greater than the previous one.

In [ ]:
def detect_overfitting(val_losses: list[float], patience: int) -> bool:
    """Check whether validation loss has been increasing for *patience* consecutive evals.

    Args:
        val_losses: List of validation loss values in chronological order.
        patience: Number of consecutive increases required to flag overfitting.

    Returns:
        ``True`` if the last *patience* validation losses are strictly
        increasing, ``False`` otherwise.
    """
    if patience <= 0 or len(val_losses) < patience + 1:
        return False
    recent = val_losses[-(patience + 1):]
    return all(recent[i] < recent[i + 1] for i in range(len(recent) - 1))

### `set_seed` — Reproducibility

Sets random seeds for Python's `random` module, NumPy, and PyTorch (both CPU
and CUDA). This ensures that training runs with the same seed and configuration
produce identical results on the same hardware.

In [ ]:
def set_seed(seed: int) -> None:
    """Set random seeds for reproducibility across Python, NumPy, and PyTorch.

    Args:
        seed: Integer seed value.
    """
    random.seed(seed)
    np.random.seed(seed)
    torch.manual_seed(seed)
    torch.cuda.manual_seed_all(seed)

### `PretrainPipeline` — Full Pretraining Loop

The `PretrainPipeline` class orchestrates the entire pretraining process:

1. **Data preparation**: Tokenise the corpus and split into train/val sets.
2. **Batch sampling**: Randomly sample contiguous chunks of `seq_len` tokens.
3. **Training loop**: For each step — sample batch, forward pass, compute cross-entropy
   loss, backward pass, update weights, adjust LR via `get_lr`.
4. **Validation**: Periodically compute validation loss on held-out data.
5. **Overfitting detection**: Warn if val loss increases for `patience` consecutive evals.
6. **Checkpoint saving**: Save model state, optimizer state, config, and losses at intervals.

| Hyperparameter | Purpose |
|---|---|
| `learning_rate` | Peak LR after warmup |
| `warmup_steps` | Steps for linear warmup |
| `max_steps` | Total training steps |
| `batch_size` | Samples per gradient update |
| `train_split` | Fraction of data for training (rest is validation) |
| `patience` | Consecutive val-loss increases before overfitting warning |
| `log_interval` | Steps between loss logging |
| `eval_interval` | Steps between validation evaluations |
| `save_interval` | Steps between checkpoint saves |

In [ ]:
class PretrainPipeline:
    """Pretraining pipeline for GPT model using next-token prediction.

    Sets up AdamW optimizer, LR schedule (warmup + cosine decay), data
    splits, and runs the training loop with checkpoint saving and
    overfitting detection.
    """

    def __init__(
        self,
        model,  # GPTModel
        config,  # TrainConfig
        tokenizer,  # BPETokenizer
    ) -> None:
        """Initialise the pretraining pipeline.

        Args:
            model: GPTModel instance to train.
            config: TrainConfig with all training hyperparameters.
            tokenizer: Trained BPETokenizer used to encode the corpus.
        """
        self.model = model
        self.config = config
        self.tokenizer = tokenizer
        self.device = next(model.parameters()).device

        # AdamW optimizer
        self.optimizer = torch.optim.AdamW(
            model.parameters(),
            lr=config.learning_rate,
            weight_decay=config.weight_decay,
            betas=(config.beta1, config.beta2),
        )

        # Loss function
        self.criterion = nn.CrossEntropyLoss()

    # ------------------------------------------------------------------
    # Data preparation
    # ------------------------------------------------------------------

    def _prepare_data(self, corpus: str) -> tuple[Tensor, Tensor]:
        """Tokenise *corpus* and split into train/val tensors.

        Returns:
            (train_ids, val_ids) — 1-D tensors of token IDs.
        """
        token_ids = self.tokenizer.encode(corpus)
        data = torch.tensor(token_ids, dtype=torch.long)
        split_idx = int(len(data) * self.config.train_split)
        return data[:split_idx], data[split_idx:]

    def _get_batch(self, data: Tensor, seq_len: int) -> tuple[Tensor, Tensor]:
        """Sample a random batch of (input, target) pairs from *data*.

        Each sample is a contiguous chunk of *seq_len* tokens; the target
        is the same chunk shifted by one position.

        Returns:
            (x, y) each of shape (batch_size, seq_len).
        """
        max_start = len(data) - seq_len - 1
        if max_start <= 0:
            x = data[:-1].unsqueeze(0)
            y = data[1:].unsqueeze(0)
            return x.to(self.device), y.to(self.device)

        indices = torch.randint(0, max_start, (self.config.batch_size,))
        x = torch.stack([data[i : i + seq_len] for i in indices])
        y = torch.stack([data[i + 1 : i + 1 + seq_len] for i in indices])
        return x.to(self.device), y.to(self.device)

    # ------------------------------------------------------------------
    # Validation
    # ------------------------------------------------------------------

    @torch.no_grad()
    def _compute_val_loss(self, val_data: Tensor, seq_len: int, num_batches: int = 5) -> float:
        """Compute average validation loss over *num_batches* random batches."""
        self.model.eval()
        total_loss = 0.0
        for _ in range(num_batches):
            x, y = self._get_batch(val_data, seq_len)
            logits = self.model(x)
            loss = self.criterion(
                logits.view(-1, logits.size(-1)),
                y.view(-1),
            )
            total_loss += loss.item()
        self.model.train()
        return total_loss / num_batches

    # ------------------------------------------------------------------
    # Checkpoint saving
    # ------------------------------------------------------------------

    def _save_checkpoint(
        self, step: int, train_losses: list[float],
        val_losses: list[float], checkpoint_dir: str,
    ) -> None:
        """Save a training checkpoint to *checkpoint_dir*."""
        os.makedirs(checkpoint_dir, exist_ok=True)
        checkpoint: dict[str, Any] = {
            "model_state_dict": self.model.state_dict(),
            "optimizer_state_dict": self.optimizer.state_dict(),
            "config": asdict(self.model.config),
            "train_config": asdict(self.config),
            "step": step,
            "train_losses": train_losses,
            "val_losses": val_losses,
            "seed": self.config.seed,
        }
        path = os.path.join(checkpoint_dir, f"checkpoint_step_{step}.pt")
        torch.save(checkpoint, path)
        logger.info("Saved checkpoint at step %d to %s", step, path)

### `PretrainPipeline.train()` — The Training Loop

The core training loop performs these operations at each step:

1. Compute the current learning rate via `get_lr` and update the optimizer.
2. Sample a random batch of (input, target) pairs from the training data.
3. Forward pass through the model to get logits.
4. Compute cross-entropy loss between predicted logits and target token IDs.
5. Backward pass to compute gradients.
6. Optimizer step to update model weights.
7. Periodically: log training loss, compute validation loss, save checkpoints.

The method accepts either raw text (`corpus`) or pre-tokenised data (`token_ids`).

In [ ]:
    # (continued on PretrainPipeline)

    def train(self, corpus: str | None = None, token_ids: list[int] | None = None) -> dict[str, list[float]]:
        """Run the pretraining loop.

        Either *corpus* (raw text) or *token_ids* (pre-tokenised) must be
        provided.  Returns a dict with ``train_losses`` and ``val_losses``.
        """
        set_seed(self.config.seed)

        # Prepare data
        if token_ids is not None:
            data = torch.tensor(token_ids, dtype=torch.long)
            split_idx = int(len(data) * self.config.train_split)
            train_data, val_data = data[:split_idx], data[split_idx:]
        elif corpus is not None:
            train_data, val_data = self._prepare_data(corpus)
        else:
            raise ValueError("Either corpus or token_ids must be provided")

        seq_len = self.model.config.max_seq_len
        checkpoint_dir = os.path.join("checkpoints", "pretrained")

        train_losses: list[float] = []
        val_losses: list[float] = []

        self.model.train()

        for step in range(1, self.config.max_steps + 1):
            # Update learning rate
            lr = get_lr(
                step, self.config.warmup_steps,
                self.config.max_steps, self.config.learning_rate,
            )
            for param_group in self.optimizer.param_groups:
                param_group["lr"] = lr

            # Forward pass
            x, y = self._get_batch(train_data, seq_len)
            logits = self.model(x)
            loss = self.criterion(
                logits.view(-1, logits.size(-1)),
                y.view(-1),
            )

            # Backward pass
            self.optimizer.zero_grad()
            loss.backward()
            self.optimizer.step()

            # Log training loss
            if step % self.config.log_interval == 0:
                train_losses.append(loss.item())
                logger.info(
                    "Step %d/%d | LR: %.6f | Train Loss: %.4f",
                    step, self.config.max_steps, lr, loss.item(),
                )

            # Evaluate on validation set
            if step % self.config.eval_interval == 0:
                val_loss = self._compute_val_loss(val_data, seq_len)
                val_losses.append(val_loss)
                logger.info(
                    "Step %d/%d | Val Loss: %.4f",
                    step, self.config.max_steps, val_loss,
                )

                # Overfitting detection
                if detect_overfitting(val_losses, self.config.patience):
                    logger.warning(
                        "Overfitting detected: validation loss increased for "
                        "%d consecutive evaluations",
                        self.config.patience,
                    )

            # Save checkpoint
            if step % self.config.save_interval == 0:
                self._save_checkpoint(step, train_losses, val_losses, checkpoint_dir)

        # Final checkpoint
        self._save_checkpoint(
            self.config.max_steps, train_losses, val_losses, checkpoint_dir
        )

        return {"train_losses": train_losses, "val_losses": val_losses}


# Attach the method to the class
PretrainPipeline.train = train

---
## Part 2 — Supervised Fine-Tuning (SFT) Pipeline

The SFT pipeline fine-tunes a pretrained GPT model on **instruction-response pairs**.
Unlike pretraining (which uses all tokens), SFT computes loss **only on the output
tokens** — the model is not penalised for the instruction portion.

### SFT Example Format

Each training example is formatted as:

```
<instruction text>
[optional input text]
\n### <output text>
```

The separator (`\n### ` by default) marks the boundary between instruction and output.
A **loss mask** is created so that tokens before the separator have mask=0 (ignored)
and tokens after have mask=1 (contribute to loss).

### Why Mask Instruction Tokens?

During fine-tuning, we want the model to learn to *generate* good outputs given
instructions. If we included instruction tokens in the loss, the model would also
be trained to predict instruction tokens — which is not the goal. Masking ensures
the gradient signal comes only from the output portion.

### `format_sft_example` — SFT Record Formatting

Concatenates the instruction, optional input, separator, and output into a single
training string. The separator token marks where the instruction ends and the
output begins — this boundary is used to create the loss mask.

In [ ]:
def format_sft_example(record: dict, separator: str) -> str:
    """Format a single SFT record into a training string.

    Concatenates instruction + optional input + separator + output.

    Args:
        record: Dict with keys ``instruction``, ``output``, and optionally ``input``.
        separator: Separator string placed between instruction/input and output.

    Returns:
        Formatted training string.
    """
    parts = [record["instruction"]]
    if record.get("input"):
        parts.append(record["input"])
    parts_str = "\n".join(parts)
    return parts_str + separator + record["output"]

### Demo: SFT Example Formatting

Let's see how an instruction-response pair gets formatted into a training string.

In [ ]:
# Demo: format an SFT example
sample_record = {
    "instruction": "Translate the following to Arabic.",
    "input": "Hello, how are you?",
    "output": "مرحبا، كيف حالك؟",
}
separator = "\n### "

formatted = format_sft_example(sample_record, separator)
print("Formatted SFT example:")
print(repr(formatted))
print()
print(formatted)

### `create_loss_mask` — Loss Masking for SFT

Creates a binary mask where instruction token positions are 0 (excluded from loss)
and output token positions are 1 (included in loss). This ensures the model is
only trained to predict the output portion of each example.

| Position | Mask Value | Meaning |
|---|---|---|
| 0 … instruction_len-1 | 0 | Instruction tokens — no loss |
| instruction_len … total_len-1 | 1 | Output tokens — loss computed |

In [ ]:
def create_loss_mask(instruction_len: int, total_len: int) -> list[int]:
    """Create a loss mask that is 0 for instruction tokens and 1 for output tokens.

    Args:
        instruction_len: Number of tokens belonging to the instruction portion.
        total_len: Total number of tokens in the sequence.

    Returns:
        List of ints (0 or 1) of length *total_len*.
    """
    return [0] * instruction_len + [1] * (total_len - instruction_len)

### Demo: Loss Mask Visualisation

Let's visualise how the loss mask looks for a sample sequence with 5 instruction
tokens and 3 output tokens.

In [ ]:
# Demo: visualise a loss mask
mask = create_loss_mask(instruction_len=5, total_len=8)
print(f"Loss mask: {mask}")
print(f"Instruction positions (mask=0): {[i for i, m in enumerate(mask) if m == 0]}")
print(f"Output positions     (mask=1): {[i for i, m in enumerate(mask) if m == 1]}")

### `SFTPipeline` — Supervised Fine-Tuning Loop

The `SFTPipeline` class handles the full SFT training process:

1. **Example preparation**: Format each SFT record, tokenise, create loss masks.
2. **Batch construction**: Pad sequences to equal length within each batch.
3. **Training loop**: Forward pass → masked cross-entropy loss → backward pass → update.
4. **Validation**: Periodically compute masked validation loss.
5. **Checkpoint saving**: Save fine-tuned model to `checkpoints/finetuned/`.

Key differences from pretraining:
- **Lower learning rate** (default 1e-5 vs 3e-4) to avoid catastrophic forgetting.
- **Loss masking** — only output tokens contribute to the gradient.
- **No LR schedule** — uses a constant learning rate (simpler for short fine-tuning runs).

In [ ]:
class SFTPipeline:
    """Supervised fine-tuning pipeline for GPT model.

    Loads a pretrained checkpoint, sets up an optimizer with a lower learning
    rate, and fine-tunes on instruction-response pairs with loss computed
    only on output tokens.
    """

    def __init__(self, model, config, tokenizer) -> None:
        """Initialise the SFT pipeline.

        Args:
            model: GPTModel instance (weights loaded from pretrained checkpoint).
            config: SFTConfig with fine-tuning hyperparameters.
            tokenizer: Trained BPETokenizer used to encode examples.
        """
        self.model = model
        self.config = config
        self.tokenizer = tokenizer
        self.device = next(model.parameters()).device

        # AdamW with lower LR than pretraining
        self.optimizer = torch.optim.AdamW(
            model.parameters(), lr=config.learning_rate,
        )

        # Loss function — reduction='none' so we can apply the mask
        self.criterion = nn.CrossEntropyLoss(reduction="none")

    # ------------------------------------------------------------------
    # Data preparation
    # ------------------------------------------------------------------

    def _prepare_examples(self, sft_data: list[dict]) -> list[tuple[list[int], list[int]]]:
        """Tokenise and prepare (token_ids, loss_mask) pairs for each SFT record."""
        examples: list[tuple[list[int], list[int]]] = []
        separator = self.config.separator_tokens
        max_seq_len = self.model.config.max_seq_len

        for record in sft_data:
            instruction_text = record["instruction"]
            if record.get("input"):
                instruction_text = instruction_text + "\n" + record["input"]
            instruction_text += separator

            instruction_ids = self.tokenizer.encode(instruction_text)
            output_ids = self.tokenizer.encode(record["output"])
            full_ids = instruction_ids + output_ids

            if len(full_ids) > max_seq_len:
                full_ids = full_ids[:max_seq_len]
                instr_len = min(len(instruction_ids), max_seq_len)
            else:
                instr_len = len(instruction_ids)

            mask = create_loss_mask(instr_len, len(full_ids))
            examples.append((full_ids, mask))
        return examples

    # ------------------------------------------------------------------
    # Batch construction
    # ------------------------------------------------------------------

    def _get_batch(self, examples, indices):
        """Build a padded batch from the given example indices.

        Returns:
            (input_ids, target_ids, loss_mask) — all of shape (batch, seq_len).
        """
        batch_tokens, batch_masks = [], []
        for idx in indices:
            tokens, mask = examples[idx]
            batch_tokens.append(tokens)
            batch_masks.append(mask)

        max_len = max(len(t) for t in batch_tokens)
        pad_id = self.tokenizer.SPECIAL_TOKENS.get("<pad>", 0)

        padded_tokens, padded_masks = [], []
        for tokens, mask in zip(batch_tokens, batch_masks):
            pad_len = max_len - len(tokens)
            padded_tokens.append(tokens + [pad_id] * pad_len)
            padded_masks.append(mask + [0] * pad_len)  # mask=0 for padding

        token_tensor = torch.tensor(padded_tokens, dtype=torch.long, device=self.device)
        mask_tensor = torch.tensor(padded_masks, dtype=torch.float, device=self.device)

        # Input is tokens[:-1], target is tokens[1:]
        input_ids = token_tensor[:, :-1]
        target_ids = token_tensor[:, 1:]
        # Shift mask to align with targets (mask[1:])
        loss_mask = mask_tensor[:, 1:]
        return input_ids, target_ids, loss_mask

### `SFTPipeline` — Validation, Checkpointing, and Training Loop

The remaining methods handle:

- **`_compute_val_loss`**: Computes average masked validation loss over all validation examples.
- **`_save_checkpoint`**: Saves model state, optimizer state, config, and losses to `checkpoints/finetuned/`.
- **`train`**: The main SFT training loop — samples batches, computes masked loss, updates weights, logs metrics, and saves checkpoints.

In [ ]:
    # (continued — SFTPipeline methods)

    @torch.no_grad()
    def _compute_val_loss(self, val_examples):
        """Compute average masked validation loss."""
        if not val_examples:
            return 0.0
        self.model.eval()
        total_loss, total_tokens = 0.0, 0
        bs = self.config.batch_size
        for start in range(0, len(val_examples), bs):
            end = min(start + bs, len(val_examples))
            indices = list(range(start, end))
            input_ids, target_ids, loss_mask = self._get_batch(val_examples, indices)
            logits = self.model(input_ids)
            loss_per_token = self.criterion(
                logits.reshape(-1, logits.size(-1)), target_ids.reshape(-1),
            ).view_as(target_ids)
            total_loss += (loss_per_token * loss_mask).sum().item()
            total_tokens += loss_mask.sum().item()
        self.model.train()
        return total_loss / total_tokens if total_tokens > 0 else 0.0

    def _save_checkpoint(self, step, train_losses, val_losses, checkpoint_dir):
        """Save a fine-tuning checkpoint."""
        os.makedirs(checkpoint_dir, exist_ok=True)
        checkpoint: dict[str, Any] = {
            "model_state_dict": self.model.state_dict(),
            "optimizer_state_dict": self.optimizer.state_dict(),
            "config": asdict(self.model.config),
            "train_config": asdict(self.config),
            "step": step,
            "train_losses": train_losses,
            "val_losses": val_losses,
            "seed": self.config.seed,
        }
        path = os.path.join(checkpoint_dir, f"sft_checkpoint_step_{step}.pt")
        torch.save(checkpoint, path)
        logger.info("Saved SFT checkpoint at step %d to %s", step, path)


# Attach methods to the class
SFTPipeline._compute_val_loss = _compute_val_loss
SFTPipeline._save_checkpoint = _save_checkpoint

### `SFTPipeline.train()` — The SFT Training Loop

The SFT training loop is similar to pretraining but with key differences:

1. **No LR schedule** — uses a constant (lower) learning rate.
2. **Masked loss** — `CrossEntropyLoss(reduction='none')` computes per-token loss,
   which is then multiplied by the loss mask before averaging.
3. **Random batch sampling** — randomly selects examples from the training set each step.
4. **90/10 train/val split** — 90% of SFT examples for training, 10% for validation.

In [ ]:
    # (continued — SFTPipeline.train)

    def train(self, sft_data: list[dict]) -> dict[str, list[float]]:
        """Run the SFT training loop.

        Fine-tunes on instruction-response pairs with loss computed only on
        output tokens. Saves checkpoints to ``checkpoints/finetuned/``.

        Args:
            sft_data: List of SFT records, each a dict with ``instruction``,
                      ``output``, and optionally ``input`` keys.

        Returns:
            ``{'train_losses': [...], 'val_losses': [...]}``
        """
        set_seed(self.config.seed)

        # Prepare all examples
        all_examples = self._prepare_examples(sft_data)

        # Split into train/val (90/10)
        split_idx = max(1, int(len(all_examples) * 0.9))
        train_examples = all_examples[:split_idx]
        val_examples = all_examples[split_idx:]

        checkpoint_dir = os.path.join("checkpoints", "finetuned")
        train_losses: list[float] = []
        val_losses: list[float] = []

        self.model.train()
        bs = self.config.batch_size

        for step in range(1, self.config.max_steps + 1):
            # Sample a random batch from training examples
            num_examples = len(train_examples)
            indices = torch.randint(0, num_examples, (min(bs, num_examples),)).tolist()

            input_ids, target_ids, loss_mask = self._get_batch(train_examples, indices)

            # Forward pass
            logits = self.model(input_ids)

            # Compute per-token loss
            loss_per_token = self.criterion(
                logits.reshape(-1, logits.size(-1)),
                target_ids.reshape(-1),
            )
            loss_per_token = loss_per_token.view_as(target_ids)

            # Apply mask: loss only on output tokens
            masked_loss = (loss_per_token * loss_mask).sum()
            num_output_tokens = loss_mask.sum()

            if num_output_tokens > 0:
                loss = masked_loss / num_output_tokens
            else:
                loss = masked_loss

            # Backward pass
            self.optimizer.zero_grad()
            loss.backward()
            self.optimizer.step()

            # Log training loss
            if step % self.config.log_interval == 0:
                train_losses.append(loss.item())
                logger.info(
                    "SFT Step %d/%d | Train Loss: %.4f",
                    step, self.config.max_steps, loss.item(),
                )

            # Evaluate on validation set
            if step % self.config.eval_interval == 0:
                val_loss = self._compute_val_loss(val_examples)
                val_losses.append(val_loss)
                logger.info(
                    "SFT Step %d/%d | Val Loss: %.4f",
                    step, self.config.max_steps, val_loss,
                )

            # Save checkpoint
            if step % self.config.save_interval == 0:
                self._save_checkpoint(step, train_losses, val_losses, checkpoint_dir)

        # Final checkpoint
        self._save_checkpoint(
            self.config.max_steps, train_losses, val_losses, checkpoint_dir
        )

        return {"train_losses": train_losses, "val_losses": val_losses}


# Attach the method to the class
SFTPipeline.train = train

---
## Part 3 — Demo: Training Loop Setup & Loss Logging

Below we demonstrate how to set up and run a short pretraining loop on
synthetic data. This shows the full pipeline in action: model creation,
pipeline initialisation, training, and loss curve plotting.

**Note**: In a real training run you would use a large corpus and many more
steps. Here we use a tiny model and few steps just to verify the pipeline works.

In [ ]:
# --- Minimal model and config for demonstration ---
# (In practice, import from src.config and src.model.transformer)

from dataclasses import dataclass


@dataclass
class GPTConfig:
    vocab_size: int = 256
    d_model: int = 64
    num_heads: int = 4
    num_layers: int = 2
    max_seq_len: int = 32
    dropout_rate: float = 0.0

    def validate(self):
        if self.d_model % self.num_heads != 0:
            raise ValueError("d_model must be divisible by num_heads")


@dataclass
class TrainConfig:
    learning_rate: float = 3e-4
    weight_decay: float = 0.01
    beta1: float = 0.9
    beta2: float = 0.999
    batch_size: int = 4
    max_steps: int = 100
    warmup_steps: int = 10
    log_interval: int = 10
    eval_interval: int = 25
    save_interval: int = 10000  # don't save during demo
    patience: int = 3
    seed: int = 42
    train_split: float = 0.9


@dataclass
class SFTConfig:
    learning_rate: float = 1e-5
    batch_size: int = 4
    max_steps: int = 50
    log_interval: int = 10
    eval_interval: int = 25
    save_interval: int = 10000  # don't save during demo
    separator_tokens: str = "\n### "
    seed: int = 42


print("Config dataclasses defined.")

### Build a Tiny GPT Model

We create a minimal GPT model (2 layers, 64-dim, 4 heads) for the demo.
This is the same architecture as `src/model/transformer.py` but with tiny
dimensions so it runs instantly.

In [ ]:
# Minimal GPT model for demo (mirrors src/model/transformer.py)

class ScaledDotProductAttention(nn.Module):
    def forward(self, Q, K, V, mask=None):
        d_k = Q.size(-1)
        scores = Q @ K.transpose(-2, -1) / math.sqrt(d_k)
        if mask is not None:
            scores = scores.masked_fill(mask == 0, float('-inf'))
        weights = torch.softmax(scores, dim=-1)
        return weights @ V


class MultiHeadAttention(nn.Module):
    def __init__(self, d_model, num_heads, dropout=0.0):
        super().__init__()
        self.num_heads = num_heads
        self.d_k = d_model // num_heads
        self.W_q = nn.Linear(d_model, d_model)
        self.W_k = nn.Linear(d_model, d_model)
        self.W_v = nn.Linear(d_model, d_model)
        self.W_o = nn.Linear(d_model, d_model)
        self.attn = ScaledDotProductAttention()
        self.dropout = nn.Dropout(dropout)

    def forward(self, x):
        B, S, _ = x.shape
        Q = self.W_q(x).view(B, S, self.num_heads, self.d_k).transpose(1, 2)
        K = self.W_k(x).view(B, S, self.num_heads, self.d_k).transpose(1, 2)
        V = self.W_v(x).view(B, S, self.num_heads, self.d_k).transpose(1, 2)
        mask = torch.tril(torch.ones(S, S, device=x.device)).unsqueeze(0).unsqueeze(0)
        out = self.attn(Q, K, V, mask)
        out = out.transpose(1, 2).contiguous().view(B, S, -1)
        return self.dropout(self.W_o(out))


class TransformerBlock(nn.Module):
    def __init__(self, d_model, num_heads, dropout=0.0):
        super().__init__()
        self.ln1 = nn.LayerNorm(d_model)
        self.attn = MultiHeadAttention(d_model, num_heads, dropout)
        self.ln2 = nn.LayerNorm(d_model)
        self.mlp = nn.Sequential(
            nn.Linear(d_model, 4 * d_model),
            nn.GELU(),
            nn.Linear(4 * d_model, d_model),
            nn.Dropout(dropout),
        )
        self.dropout = nn.Dropout(dropout)

    def forward(self, x):
        x = x + self.dropout(self.attn(self.ln1(x)))
        x = x + self.mlp(self.ln2(x))
        return x


class GPTModel(nn.Module):
    def __init__(self, config: GPTConfig):
        super().__init__()
        self.config = config
        self.token_emb = nn.Embedding(config.vocab_size, config.d_model)
        self.pos_emb = nn.Embedding(config.max_seq_len, config.d_model)
        self.blocks = nn.ModuleList([
            TransformerBlock(config.d_model, config.num_heads, config.dropout_rate)
            for _ in range(config.num_layers)
        ])
        self.ln_f = nn.LayerNorm(config.d_model)
        self.lm_head = nn.Linear(config.d_model, config.vocab_size, bias=False)

    def forward(self, token_ids):
        B, S = token_ids.shape
        if S > self.config.max_seq_len:
            raise ValueError(f"Sequence length {S} exceeds max_seq_len {self.config.max_seq_len}")
        pos = torch.arange(S, device=token_ids.device).unsqueeze(0)
        x = self.token_emb(token_ids) + self.pos_emb(pos)
        for block in self.blocks:
            x = block(x)
        x = self.ln_f(x)
        return self.lm_head(x)

    def count_parameters(self):
        return sum(p.numel() for p in self.parameters() if p.requires_grad)


gpt_config = GPTConfig()
model = GPTModel(gpt_config)
print(f"Model parameters: {model.count_parameters():,}")

### Run a Short Pretraining Demo

We create a mock tokenizer and run 100 steps of pretraining on random token IDs.
This demonstrates the full training loop: LR scheduling, loss logging, validation,
and overfitting detection.

In [ ]:
# Mock tokenizer for demo (just needs encode/decode)
class MockTokenizer:
    SPECIAL_TOKENS = {'<pad>': 0, '<bos>': 1, '<eos>': 2, '<unk>': 3}
    def encode(self, text: str) -> list[int]:
        return [b for b in text.encode('utf-8')]
    def decode(self, ids: list[int]) -> str:
        return bytes(ids).decode('utf-8', errors='replace')

tokenizer = MockTokenizer()

# Generate synthetic token IDs (simulating a tokenised corpus)
set_seed(42)
synthetic_ids = torch.randint(4, 256, (2000,)).tolist()

# Set up and run pretraining
train_config = TrainConfig()
pipeline = PretrainPipeline(model, train_config, tokenizer)

logging.basicConfig(level=logging.WARNING)  # reduce noise for demo
results = pipeline.train(token_ids=synthetic_ids)

print(f'Training steps completed: {train_config.max_steps}')
print(f'Train losses logged: {len(results["train_losses"])}')
print(f'Val losses logged:   {len(results["val_losses"])}')

### Plot Training & Validation Loss Curves

Visualising the loss curves is essential for monitoring training progress.
A healthy training run shows both curves decreasing. If validation loss
starts increasing while training loss continues to decrease, the model
is overfitting.

In [ ]:
# Plot loss curves from the demo run
fig, (ax1, ax2) = plt.subplots(1, 2, figsize=(12, 4))

# Training loss
ax1.plot(results['train_losses'], label='Train Loss')
ax1.set_xlabel('Log Step')
ax1.set_ylabel('Loss')
ax1.set_title('Training Loss')
ax1.legend()

# Validation loss
if results['val_losses']:
    ax2.plot(results['val_losses'], label='Val Loss', color='orange')
    ax2.set_xlabel('Eval Step')
    ax2.set_ylabel('Loss')
    ax2.set_title('Validation Loss')
    ax2.legend()
else:
    ax2.text(0.5, 0.5, 'No validation losses logged',
             ha='center', va='center', transform=ax2.transAxes)

plt.tight_layout()
plt.show()

---
## Summary

This notebook implemented the two training pipelines for the GPT model:

| Pipeline | Source | Purpose |
|---|---|---|
| `PretrainPipeline` | `src/training/pretrain.py` | Train from scratch on raw text via next-token prediction |
| `SFTPipeline` | `src/training/finetune.py` | Fine-tune on instruction-response pairs with masked loss |

### Key components covered

- `get_lr()` — Linear warmup + cosine decay learning rate schedule
- `detect_overfitting()` — Consecutive validation loss increase detection
- `set_seed()` — Reproducibility across Python, NumPy, and PyTorch
- `format_sft_example()` — SFT record formatting with separator tokens
- `create_loss_mask()` — Binary mask for instruction vs output tokens
- Checkpoint saving with full config for reproducibility

### Next steps

- **Notebook 07**: Text generation with greedy, top-k, top-p, and temperature sampling
- **Notebook 08**: Evaluation — perplexity, sample generation, loss curves, error analysis